# EAGLE · Cuadrícula territorial de 10 × 10 km — Península Ibérica

**Objetivo:** crear una **tabla maestra geográfica** con **una fila por celda de 100 km²**, identificador estable, punto representativo y columnas listas para añadir las variables del proyecto EAGLE.

**Ámbito:** España y Portugal **continentales**, Andorra y Gibraltar (si está presente en la capa pública). No incluye Baleares, Canarias, Madeira ni Azores. Las celdas que tocan la costa o una frontera **se conservan completas**, pero también calculamos cuánta superficie terrestre contienen.

**Metodología:** se proyectan los límites nacionales a **ETRS89 / LAEA Europe (EPSG:3035)**, un sistema métrico de área equivalente; se crea una malla regular con cuadrados de **10.000 × 10.000 m**. Para cada celda se genera un punto representativo situado en tierra incluso cuando su centro geométrico cae en el mar. La malla es **geográfica**, NO añade buses o líneas eléctricos ficticios a PyPSA-Eur.

**Archivos de salida:** CSV para Pandas/Excel y GeoPackage para QGIS/GeoPandas, con cuadrículas y puntos.

## 0 · Preparación del entorno

En una terminal de Anaconda, activa el entorno donde tienes PyPSA-Eur e instala (si hace falta):

```bash
conda install -c conda-forge geopandas shapely pyproj pyogrio requests matplotlib pandas numpy jupyter -y
```

La primera ejecución de este notebook **requiere acceso a Internet** para descargar los límites de Natural Earth (resolución 10m). Después se usa una copia local en la carpeta `datos_fuente/`.

In [ ]:
from pathlib import Path
import math
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
from shapely.ops import unary_union
from shapely.validation import explain_validity
import requests

# CONFIGURACIÓN DEL PROYECTO
CARPETA_SALIDA = Path("eagle_grid_10km")
CARPETA_FUENTE = CARPETA_SALIDA / "datos_fuente"
CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)
CARPETA_FUENTE.mkdir(parents=True, exist_ok=True)

LADO_CELDA_M = 10_000
CRS_METRICO = "EPSG:3035"
CRS_GPS = "EPSG:4326"
# Si se define > 0, excluye celdas costeras con una proporción terrestre inferior al umbral.
UMBRAL_TIERRA = 0.0

print("GeoPandas:", gpd.__version__)
print("Destino:", CARPETA_SALIDA.resolve())

## 1 · Descargar o cargar los límites de países

Utilizamos la capa pública **Natural Earth Admin 0, escala 1:10m**. La copia local permite repetir el proceso sin descargarla otra vez. Si tu red bloquea el enlace, descarga manualmente el ZIP de [Natural Earth](https://www.naturalearthdata.com/downloads/10m-cultural-vectors/10m-admin-0-countries/) y guárdalo en `eagle_grid_10km/datos_fuente/ne_10m_admin_0_countries.zip`.

In [ ]:
URL_NATURAL_EARTH = "https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_admin_0_countries.zip"
ZIP_PAISES = CARPETA_FUENTE / "ne_10m_admin_0_countries.zip"

if not ZIP_PAISES.exists():
    print("Descargando límites nacionales (solo primera vez)...")
    try:
        with requests.get(URL_NATURAL_EARTH, timeout=90, stream=True) as r:
            r.raise_for_status()
            with open(ZIP_PAISES, "wb") as f:
                for trozo in r.iter_content(chunk_size=1024 * 1024):
                    if trozo:
                        f.write(trozo)
    except requests.RequestException as exc:
        ZIP_PAISES.unlink(missing_ok=True)
        raise RuntimeError(
            "No se pudo descargar Natural Earth. Descarga el ZIP manualmente "
            f"en {ZIP_PAISES.resolve()} y ejecuta la celda otra vez."
        ) from exc

paises = gpd.read_file(f"zip://{ZIP_PAISES.resolve()}")
print(f"Países/territorios leídos: {len(paises):,}")
print("Columnas disponibles:", paises.columns[:20].tolist())

## 2 · Delimitar únicamente la parte continental ibérica

Seleccionamos España, Portugal, Andorra y Gibraltar por códigos ISO/nombres. Para España y Portugal **retenemos el polígono continuo de mayor superficie**, evitando islas que se encuentren dentro de la extensión del mapa. Esto es una aproximación territorial y no sustituye límites administrativos oficiales de precisión catastral.

In [ ]:
# Buscamos por varias columnas para adaptarnos a revisiones de Natural Earth.
CODIGOS = {"ESP": "ES", "PRT": "PT", "AND": "AD", "GIB": "GI"}
NOMBRES = {"Spain": "ES", "Portugal": "PT", "Andorra": "AD", "Gibraltar": "GI"}

seleccion = []
for _, fila in paises.iterrows():
    iso = str(fila.get("ADM0_A3", ""))
    nombre = str(fila.get("ADMIN", ""))
    codigo = CODIGOS.get(iso, NOMBRES.get(nombre))
    if codigo:
        geom = fila.geometry
        if geom is None or geom.is_empty:
            continue
        # Normalizamos multipartes; extraemos por superficie el mayor polígono continental.
        partes = list(geom.geoms) if geom.geom_type == "MultiPolygon" else [geom]
        partes = [g for g in partes if g.geom_type == "Polygon" and not g.is_empty]
        if not partes:
            continue
        continente = max(partes, key=lambda p: p.area)
        seleccion.append({"pais": codigo, "geometry": continente})

territorios = gpd.GeoDataFrame(seleccion, geometry="geometry", crs=paises.crs)
if not {"ES", "PT"}.issubset(set(territorios["pais"])):
    raise ValueError("No se encontraron España y Portugal en Natural Earth. Revisa columnas/códigos.")

# Evitar repetidos si el dataset contiene versiones duplicadas del territorio.
territorios = territorios.to_crs(CRS_METRICO).dissolve(by="pais", as_index=False)
# Algunos registros podrían tener geometrías no válidas.
territorios["geometry"] = territorios.geometry.make_valid()

# Unión geográfica para filtrar celdas que tocan tierra firme.
area_iberica = unary_union(territorios.geometry.tolist())
assert not area_iberica.is_empty

print("Territorios incluidos:", sorted(territorios.pais.tolist()))
print(f"Superficie representada: {area_iberica.area / 1e6:,.0f} km²")
fig, ax = plt.subplots(figsize=(9, 7))
territorios.plot(ax=ax, column="pais", legend=True, edgecolor="black", linewidth=.4)
ax.set_aspect("equal"); ax.set_title("Área territorial de estudio · Iberia continental")
plt.show()

## 3 · Construir cuadrados de exactamente 10 × 10 km

La malla se alinea con los múltiplos de 10.000 m de EPSG:3035, de modo que los identificadores se mantengan estables al ampliar el área en el futuro. Conservamos una celda si tiene **intersección con tierra de área positiva**, incluso si una parte del cuadrado cae fuera de la Península.

In [ ]:
xmin, ymin, xmax, ymax = area_iberica.bounds
paso = LADO_CELDA_M
x_inicio = math.floor(xmin / paso) * paso
y_inicio = math.floor(ymin / paso) * paso
x_final = math.ceil(xmax / paso) * paso
y_final = math.ceil(ymax / paso) * paso

celdas = []
for x in range(x_inicio, x_final, paso):
    for y in range(y_inicio, y_final, paso):
        celdas.append({
            "columna_malla": x // paso,
            "fila_malla": y // paso,
            "geometry": box(x, y, x + paso, y + paso),
        })

malla = gpd.GeoDataFrame(celdas, geometry="geometry", crs=CRS_METRICO)
# SJOIN usa el índice espacial, evita calcular intersecciones para celdas lejanas.
tocan_territorio = gpd.sjoin(
    malla, territorios[["pais", "geometry"]], how="inner", predicate="intersects"
)
malla = malla.loc[tocan_territorio.index.unique()].copy().reset_index(drop=True)

# Superficie de tierra por celda. La geometría original sigue siendo un cuadrado completo.
recortes = malla.geometry.intersection(area_iberica)
malla["superficie_tierra_km2"] = recortes.area / 1_000_000
malla = malla[malla["superficie_tierra_km2"] > 0].copy()
malla["fraccion_tierra"] = malla["superficie_tierra_km2"] / (paso * paso / 1e6)
malla = malla[malla.fraccion_tierra >= UMBRAL_TIERRA].copy().reset_index(drop=True)
# Recalcular recortes con el índice definitivo para asegurar una alineación correcta.
recortes = malla.geometry.intersection(area_iberica)

malla["celda_id"] = malla.apply(
    lambda r: f"EAGLE10_{int(r.columna_malla):04d}_{int(r.fila_malla):04d}", axis=1
)
assert malla.celda_id.is_unique

print(f"Celdas seleccionadas: {len(malla):,}")
print(f"Área nominal por celda: {paso*paso/1e6:.0f} km²")
print(f"Celdas de borde: {(malla.fraccion_tierra < .999).sum():,}")

## 4 · Localizar un punto terrestre representativo en cada cuadrado

El centro del cuadrado puede estar sobre el mar en zonas costeras. Por ello ofrecemos ambas coordenadas: **centro geométrico** y **punto representativo de la intersección terrestre**. Para conectar datos geográficos utiliza normalmente `lon_punto`, `lat_punto`; para operaciones de cuadrícula utiliza la geometría de los cuadrados.

In [ ]:
# El representante está garantizado dentro de la geometría terrestre intersectada.
puntos_tierra = recortes.loc[malla.index].representative_point()
centros = malla.geometry.centroid

puntos_tierra_gps = gpd.GeoSeries(puntos_tierra, crs=CRS_METRICO).to_crs(CRS_GPS)
centros_gps = gpd.GeoSeries(centros, crs=CRS_METRICO).to_crs(CRS_GPS)

malla["lon_punto"] = puntos_tierra_gps.x.to_numpy()
malla["lat_punto"] = puntos_tierra_gps.y.to_numpy()
malla["lon_centro"] = centros_gps.x.to_numpy()
malla["lat_centro"] = centros_gps.y.to_numpy()
malla["x_centro_m"] = centros.x.to_numpy()
malla["y_centro_m"] = centros.y.to_numpy()

# País predominante por área terrestre dentro de cada celda.
cruces = gpd.sjoin(malla[["celda_id", "geometry"]], territorios, predicate="intersects", how="inner")
cruces["area_interseccion_m2"] = cruces.apply(
    lambda r: malla.loc[r.name, "geometry"].intersection(
        territorios.loc[r.index_right, "geometry"]
    ).area,
    axis=1,
)
mejores = cruces.sort_values("area_interseccion_m2", ascending=False).drop_duplicates("celda_id")
malla = malla.merge(mejores[["celda_id", "pais"]], on="celda_id", how="left")

columnas_principales = [
    "celda_id", "pais", "columna_malla", "fila_malla", "lon_punto", "lat_punto",
    "lon_centro", "lat_centro", "x_centro_m", "y_centro_m",
    "superficie_tierra_km2", "fraccion_tierra"
]
malla = malla[columnas_principales + ["geometry"]]
assert not malla["pais"].isna().any()
print(malla[columnas_principales].head(8).to_string(index=False))

## 5 · Añadir columnas listas para los indicadores EAGLE

Dejamos los valores como `NaN`, **no como ceros**, ya que todavía no hemos calculado esos indicadores. Cada dato debe medirse o estimarse posteriormente desde su propia fuente. La capacidad de conexión y la congestión de la red **no pueden deducirse únicamente por proximidad** a un bus.

In [ ]:
variables_eagle = {
    # ELECTRICIDAD
    "subestacion_mas_cercana_id": "string",
    "distancia_subestacion_km": "float",
    "tension_subestacion_kv": "float",
    "capacidad_conexion_estimada_mw": "float",
    "cargabilidad_red_pct": "float",
    "coste_conexion_estimado_eur": "float",
    # AGUA
    "disponibilidad_hidrica_indice": "float",
    "estres_hidrico_indice": "float",
    "distancia_recurso_hidrico_km": "float",
    # RENOVABLES / CLIMA
    "recurso_solar_kwh_m2_anio": "float",
    "recurso_eolico_m_s": "float",
    "temperatura_media_c": "float",
    # SUELO Y CONECTIVIDAD
    "suelo_disponible_ha": "float",
    "coste_suelo_eur_m2": "float",
    "distancia_fibra_km": "float",
    # RIESGOS / ENTORNO
    "riesgo_inundacion_indice": "float",
    "riesgo_incendio_indice": "float",
    "zona_protegida_pct": "float",
    # EVALUACIÓN MULTICRITERIO
    "apto_restricciones": "boolean",
    "score_eagle": "float",
}
for nombre, tipo in variables_eagle.items():
    malla[nombre] = (pd.Series(pd.NA, index=malla.index, dtype="string")
                     if tipo == "string" else
                     pd.Series(pd.NA, index=malla.index, dtype="boolean")
                     if tipo == "boolean" else
                     np.nan)

print(f"Filas: {len(malla):,} | Columnas (incl. geometría): {len(malla.columns)}")
display(malla.drop(columns="geometry").head())

## 6 · Visualizar el resultado

En todo el territorio verás una cuadrícula muy densa. El gráfico mantiene el límite continental y permite comprobar que no hemos creado una nueva red eléctrica artificial.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
territorios.plot(ax=ax, color="#e6f0e5", edgecolor="#365d40", linewidth=0.8)
malla.boundary.plot(ax=ax, linewidth=0.14, color="#3877b4", alpha=.65)
ax.set_title(f"EAGLE | Malla territorial de 10 × 10 km | {len(malla):,} celdas")
ax.set_axis_off(); ax.set_aspect("equal")
plt.tight_layout()
plt.savefig(CARPETA_SALIDA / "mapa_malla_10km.png", dpi=200, bbox_inches="tight")
plt.show()

## 7 · Exportar tabla CSV y geometrías GeoPackage

- **`eagle_celdas_10km.csv`**: tabla maestra que puedes cargar con Pandas y enriquecer mediante `celda_id`.
- **`eagle_celdas_10km.gpkg`**: cuadrados completos (`celdas_10km`) y puntos representativos terrestres (`puntos_celdas`) para usar en QGIS/GeoPandas.
- **`mapa_malla_10km.png`**: vista de comprobación.

Los sistemas de referencia se guardan dentro del GeoPackage. El CSV tiene coordenadas WGS84 (longitud, latitud).

In [ ]:
ruta_csv = CARPETA_SALIDA / "eagle_celdas_10km.csv"
ruta_gpkg = CARPETA_SALIDA / "eagle_celdas_10km.gpkg"

malla.drop(columns="geometry").to_csv(ruta_csv, index=False, encoding="utf-8-sig")

# Al exportar usamos float/objetos compatibles con OGR y dejamos los datos ausentes nulos.
malla_export = malla.copy()
malla_export["subestacion_mas_cercana_id"] = malla_export["subestacion_mas_cercana_id"].astype(object)
malla_export["apto_restricciones"] = malla_export["apto_restricciones"].astype(object)

ruta_gpkg.unlink(missing_ok=True) # Regenerar limpiamente en cada ejecución.
malla_export.to_file(ruta_gpkg, layer="celdas_10km", driver="GPKG")

puntos_export = malla_export.copy()
puntos_export = puntos_export.set_geometry(
    gpd.GeoSeries(puntos_tierra.loc[malla.index], index=malla.index, crs=CRS_METRICO)
)
puntos_export.to_file(ruta_gpkg, layer="puntos_celdas", driver="GPKG", mode="a")

print("Archivos generados:")
for fichero in [ruta_csv, ruta_gpkg, CARPETA_SALIDA / "mapa_malla_10km.png"]:
    print(" •", fichero.resolve(), f"({fichero.stat().st_size / 1e6:.2f} MB)")


## 8 · Validaciones y ejemplo de uso posterior

Comprobamos IDs únicos, coordenadas y extensión de celdas. A partir de ahora cualquier dataset externo se puede cruzar por coordenadas/solape geográfico y unirse por `celda_id`. Nunca sobrescribas un `NaN` con 0 salvo que una observación realmente signifique ausencia del recurso.

In [ ]:
assert malla.celda_id.nunique() == len(malla)
assert malla.crs.to_string() == CRS_METRICO
assert malla.fraccion_tierra.between(0, 1.00001).all()
assert malla["lon_punto"].between(-11, 5).all()
assert malla["lat_punto"].between(35, 45).all()

print("OK · Identificadores únicos y datos espaciales válidos")
print("Celdas por país predominante:")
print(malla.groupby("pais").size().rename("num_celdas"))

# Ejemplo para el siguiente notebook:
# df = pd.read_csv(ruta_csv)
# df_recursos = pd.DataFrame({"celda_id": [...], "recurso_solar_kwh_m2_anio": [...]})
# df = df.drop(columns=["recurso_solar_kwh_m2_anio"]).merge(df_recursos, on="celda_id", how="left")

---
### Notas para el desarrollo de EAGLE

- El área cubierta en la costa corresponde a **celdas completas**, pero `superficie_tierra_km2` describe la parte continental realmente contenida en cada una. En fases futuras podría ser más apropiado usar la **geometría recortada a tierra** para estadísticas zonales (agua, radiación, suelo, etc.).
- Esta tabla es un **inventario de evaluación territorial**, no una tabla de conexiones aprobadas. Los valores eléctricos deben calcularse o consultarse con fuentes verificables y escenarios definidos.
- Para análisis posteriores puede ser útil guardar también el año/escenario, la fuente, la fecha y la incertidumbre de cada variable.
- Los límites de Natural Earth son cartografía general, no sustituyen a Catastro, planeamiento o límites regulatorios detallados.